# 추론 모델 밑바닥부터 2편 실습 노트북 — 3장: 추론 모델 평가하기(채점기 만들기) (한국어 주석판)

**출처:** 원본 노트북 `ch03_main.ipynb`, Sebastian Raschka, https://github.com/rasbt/reasoning-from-scratch , Apache License 2.0. 책 *Build a Reasoning Model (From Scratch)*, Manning.

**변경 사항:** 코난쌤(@conanssam)이 설명 글을 한국어로 옮기고 코드에 한국어 주석을 덧붙였습니다. 코드 동작은 원본과 같습니다.

**쓰는 법**
1. 저장소를 내려받습니다: `git clone https://github.com/rasbt/reasoning-from-scratch.git`
2. 저장소 폴더에서 `uv sync`를 실행해 필요한 패키지를 설치합니다.
3. `uv run jupyter lab`으로 주피터랩을 열고, 이 파일을 `ch03/01_main-chapter-code/` 폴더에 넣어 실행합니다.

**팁:** MATH-500 전체 평가는 오래 걸립니다. 처음에는 원본처럼 앞 10문항(`math_data[:10]`)으로 돌려 보세요.

<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://sebastianraschka.com">Sebastian Raschka</a>의 책 <a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a>에 딸린 실습 코드입니다<br>
<br>코드 저장소: <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 3장: 추론 모델 평가하기

In [ ]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "sympy",
    "tokenizers"  # Used by reasoning_from_scratch
]

# 설치된 패키지 버전을 하나씩 출력합니다
for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

<br>

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch03/CH03_F01_raschka.webp?1" width="500px">

&nbsp;
## 3.1 수학 채점기 만들기

- 이 절에는 코드가 없습니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch03/CH03_F02_raschka.webp?1" width="500px">

<br>
<br>
<br>

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch03/CH03_F03_raschka.webp?1" width="500px">

&nbsp;
## 3.2 사전학습된 모델을 불러와 글 생성하기

- 이 절에서는 평가할 모델을 불러옵니다(2장 복습입니다).

In [ ]:
from pathlib import Path
import torch


from reasoning_from_scratch.qwen3 import (
    download_qwen3_small,
    Qwen3Tokenizer,
    Qwen3Model,
    QWEN_CONFIG_06_B
)


# 고른 모델(base 또는 reasoning)과 짝 토크나이저를 불러오는 함수입니다
def load_model_and_tokenizer(
    which_model, device, use_compile, local_dir="qwen3"
):
    if which_model == "base":

        # 기본(base) 모델 가중치와 토크나이저를 qwen3/ 폴더에 내려받습니다
        download_qwen3_small(
            kind="base", tokenizer_only=False, out_dir=local_dir
        )

        tokenizer_path = Path(local_dir) / "tokenizer-base.json"
        model_path = Path(local_dir) / "qwen3-0.6B-base.pth"
        tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

    # 추론 모델은 채팅 템플릿과 생각(thinking) 모드를 켠 토크나이저를 씁니다
    elif which_model == "reasoning":

        download_qwen3_small(
            kind="reasoning", tokenizer_only=False, out_dir=local_dir
        )

        tokenizer_path = Path(local_dir) / "tokenizer-reasoning.json"
        model_path = Path(local_dir) / "qwen3-0.6B-reasoning.pth"
        tokenizer = Qwen3Tokenizer(
            tokenizer_file_path=tokenizer_path,
            apply_chat_template=True,
            add_generation_prompt=True,
            add_thinking=True,
        )

    else:
        raise ValueError(f"Invalid choice: which_model={which_model}")

    # 모델 뼈대를 만들고 내려받은 가중치를 채웁니다
    model = Qwen3Model(QWEN_CONFIG_06_B)
    model.load_state_dict(torch.load(model_path))

    model.to(device)

    # 원하면 torch.compile로 모델을 컴파일합니다
    if use_compile:
        torch._dynamo.config.allow_unspec_int_on_nn_module = True
        model = torch.compile(model)

    return model, tokenizer

- 여기서는 기본 모델(base model)을 씁니다. 이 장을 다 마친 뒤에는 `which_model="base"`를 `which_model="reasoning"`으로 바꾸고 노트북을 다시 돌려 보세요. 이미 학습된 추론 모델(reasoning model)을 평가할 수 있습니다.

In [ ]:
from reasoning_from_scratch.ch02 import (
    get_device
)

# 평가할 모델을 고릅니다 ("base" 또는 "reasoning")
WHICH_MODEL = "base"
device = get_device()

# If you have compatibility issues, try to
# uncomment the line below and rerun the notebook
# device = torch.device("cpu")

# 모델과 토크나이저를 불러옵니다
model, tokenizer = load_model_and_tokenizer(
    which_model=WHICH_MODEL,
    device=device,
    use_compile=False
)

In [ ]:
from reasoning_from_scratch.ch02 import (
    generate_text_basic_stream_cache
)

# 모델에게 풀게 할 수학 문제입니다 (LaTeX 수식으로 적혀 있습니다)
prompt = (
    r"If $a+b=3$ and $ab=\tfrac{13}{6}$, "
    r"what is the value of $a^2+b^2$?"
)

# Similar to chapter 2 exercise solution:
input_token_ids_tensor = torch.tensor(
    tokenizer.encode(prompt),
    device=device
    ).unsqueeze(0)

# 토큰을 하나 만들 때마다 바로 화면에 출력합니다
all_token_ids = []
for token in generate_text_basic_stream_cache(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=2048,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0)
    decoded_id = tokenizer.decode(token_id.tolist())
    print(
        decoded_id,
        end="",
        flush=True
    )
    all_token_ids.append(token_id)

# 만든 토큰 전체를 글 하나로 되돌립니다
all_tokens = tokenizer.decode(all_token_ids)

- LaTeX 문법이 낯설다면 위 답은 읽기 아주 어려울 거예요.
- 아래처럼 `Latex` 클래스를 쓰면 LaTeX 문법을 수식으로 그려 주어서 훨씬 읽기 쉬워집니다.

In [ ]:
from IPython.display import Latex, display

# 모델 답을 LaTeX 수식으로 보기 좋게 그려 줍니다
display(Latex(all_tokens))

- 특정 수식만 그리고 싶다면 `Math` 클래스를 써도 됩니다.

In [ ]:
from IPython.display import Math

# 수식 하나만 골라 그려 봅니다
display(Math(r"\dfrac{14}{3}"))

&nbsp;
## 3.3 글 생성을 쉽게 해 주는 감싸기 함수 만들기

- 위에서 사전학습된 LLM을 불러오고 글 생성 기능을 준비했습니다(아래 그림 참고). 이 두 가지가 이 장 나머지에서 다룰 평가 과정의 첫 두 단계입니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch03/CH03_F05_raschka.webp?1" width="500px">

- 더 편하게 쓰려고 글 생성 함수를 감싸는 함수(wrapper)를 만듭니다. 이제 모델, 토크나이저, 프롬프트와 몇 가지 설정만 넣으면 됩니다.

In [ ]:
# 모델·토크나이저·프롬프트만 넣으면 생성된 글 전체를 돌려주는 감싸기(wrapper) 함수입니다
def generate_text_stream_concat(
    model, tokenizer, prompt, device, max_new_tokens,
    verbose=False,
):
    input_ids = torch.tensor(
        tokenizer.encode(prompt), device=device
        ).unsqueeze(0)

    generated_ids = []
    for token in generate_text_basic_stream_cache(
        model=model,
        token_ids=input_ids,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id,
    ):
        next_token_id = token.squeeze(0)
        # 새 토큰 ID를 목록에 모아 둡니다
        generated_ids.append(next_token_id.item())

        # verbose=True면 만드는 대로 화면에도 보여 줍니다
        if verbose:
            print(
                tokenizer.decode(next_token_id.tolist()),
                end="",
                flush=True
            )
    # 모은 토큰 ID를 글로 바꿔 돌려줍니다
    return tokenizer.decode(generated_ids)


# True로 바꾸면 아래 생성 단계를 건너뜁니다
skip_portion = False

if not skip_portion:
    generated_text = generate_text_stream_concat(
        model, tokenizer, prompt, device,
        max_new_tokens=2048,
        verbose=True
    )

&nbsp;
## 3.4 최종 답 상자 꺼내기

- 이 절에서는 답 상자(answer box)를 꺼냅니다(3단계). 다음 절에서는 꺼낸 답을 정규화(normalize)합니다(4단계).

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch03/CH03_F06_raschka.webp?1" width="500px">

In [ ]:
# 모델 답을 흉내 낸 예시 글입니다 (끝에 \boxed{...} 답 상자가 있습니다)
model_answer = (
r"""... some explanation...
**Final Answer:**

\[
\boxed{\dfrac{14}{3}}
\]
""")

In [ ]:
# 글에서 마지막 \boxed{...} 안의 내용을 꺼내는 함수입니다
def get_last_boxed(text):
    # Find the last occurrence of "\boxed"
    boxed_start_idx = text.rfind(r"\boxed")
    if boxed_start_idx == -1:
        return None

    # Get position after "\boxed"
    current_idx = boxed_start_idx + len(r"\boxed")

    # Skip any whitespace after "\boxed"
    while current_idx < len(text) and text[current_idx].isspace():
        current_idx += 1

    # Expect an opening brace "{"
    if current_idx >= len(text) or text[current_idx] != "{":
        return None

    # Parse the braces with nesting
    current_idx += 1
    # 중괄호 깊이를 세면서 짝이 맞는 닫는 괄호를 찾습니다
    brace_depth = 1
    content_start_idx = current_idx

    while current_idx < len(text) and brace_depth > 0:
        char = text[current_idx]
        if char == "{":
            brace_depth += 1
        elif char == "}":
            brace_depth -= 1
        current_idx += 1

    # Account for unbalanced braces
    if brace_depth != 0:
        return None

    # Extract content inside the outermost braces
    return text[content_start_idx:current_idx-1]

In [ ]:
# 예시 답에서 답 상자 내용을 꺼내 봅니다
extracted_answer = get_last_boxed(model_answer)
print(extracted_answer)

In [ ]:
import re

# 음수·분수·소수·지수 표기 숫자를 찾는 정규식입니다
RE_NUMBER = re.compile(
    r"-?(?:\d+/\d+|\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)"
)

# 답 상자가 있으면 그 내용을, 없으면 fallback 규칙대로 답 후보를 고릅니다
def extract_final_candidate(text, fallback="number_then_full"):
    # Default return value if nothing matches
    result = ""

    if text:
        # Prefer the last boxed expression if present
        boxed = get_last_boxed(text.strip())
        if boxed:
            result = boxed.strip().strip("$ ")

        # If no boxed expression, try fallback
        elif fallback in ("number_then_full", "number_only"):
            m = RE_NUMBER.findall(text)
            if m:
                # Use last number
                result = m[-1]
            elif fallback == "number_then_full":
                # Else return full text if no number found
                result = text
    return result

- 답 상자 내용이 없을 때 쓰는 fallback 설정입니다.
    - "number_then_full": 마지막에 나온 단순한 숫자를 고릅니다. 숫자가 없으면 글 전체를 고릅니다.
    - "number_only": 마지막에 나온 단순한 숫자를 고릅니다. 숫자가 없으면 빈 문자열 `""`을 돌려줍니다.
    - `None`: 답 상자 내용만 꺼냅니다. 없으면 빈 문자열 `""`을 돌려줍니다.

In [ ]:
print(extract_final_candidate(model_answer))

In [ ]:
# 답 상자만 있는 글
print(extract_final_candidate(r"\boxed{ 14/3. }"))

In [ ]:
# 답 상자가 없으면 마지막 숫자를 고릅니다
print(extract_final_candidate("abc < > 14/3 abc"))

In [ ]:
# 숫자도 없으면 글 전체를 돌려줍니다 (기본 fallback)
print(extract_final_candidate("Text without numbers"))

&nbsp;
## 3.5 꺼낸 답 정규화하기

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch03/CH03_F07_raschka.webp?1" width="500px">

- 앞 절에서 답을 꺼냈습니다(3단계). 이제 그 답을 정규화합니다(앞 그림의 4단계).

In [ ]:
# (찾을 패턴, 바꿀 글) 쌍 목록입니다
LATEX_FIXES = [  # Latex formatting to be replaced
    (r"\\left\s*", ""),
    (r"\\right\s*", ""),
    (r"\\,|\\!|\\;|\\:", ""),
    (r"\\cdot", "*"),
    (r"\u00B7|\u00D7", "*"),
    (r"\\\^\\circ", ""),
    (r"\\dfrac", r"\\frac"),
    (r"\\tfrac", r"\\frac"),
    (r"°", ""),
]

RE_SPECIAL = re.compile(r"<\|[^>]+?\|>")  # strip chat special tokens like <|assistant|>

# 꺼낸 답을 비교하기 쉬운 한 가지 모양으로 다듬는 함수입니다
def normalize_text(text):
    if not text:
        return ""
    text = RE_SPECIAL.sub("", text).strip()
    # 위 첨자(²·³ 등)를 보통 글자로 바꾸는 표입니다
    SUPERSCRIPT_MAP = {
        "⁰": "0", "¹": "1", "²": "2", "³": "3", "⁴": "4",
        "⁵": "5", "⁶": "6", "⁷": "7", "⁸": "8", "⁹": "9",
        "⁺": "+", "⁻": "-", "⁽": "(", "⁾": ")",
    }

    # Strip leading multiple-choice labels
    # E.g., like "c. 3" -> 3, or "b: 2" -> 2
    match = re.match(r"^[A-Za-z]\s*[.:]\s*(.+)$", text)
    if match:
        text = match.group(1)
        
    # Remove angle-degree markers
    text = re.sub(r"\^\s*\{\s*\\circ\s*\}", "", text)   # ^{\circ}
    text = re.sub(r"\^\s*\\circ", "", text)             # ^\circ
    text = text.replace("°", "")                        # Unicode degree

    # unwrap \text{...} if the whole string is wrapped
    match = re.match(r"^\\text\{(?P<x>.+?)\}$", text)
    if match:
        text = match.group("x")

    # strip inline/display math wrappers \( \) \[ \]
    text = re.sub(r"\\\(|\\\)|\\\[|\\\]", "", text)

    # light LaTeX canonicalization
    for pat, rep in LATEX_FIXES:
        text = re.sub(pat, rep, text)

    def convert_superscripts(s, base=None):
        converted = "".join(
            SUPERSCRIPT_MAP[ch] if ch in SUPERSCRIPT_MAP else ch
            for ch in s
        )
        if base is None:
            return converted
        return f"{base}**{converted}"

    # convert unicode superscripts into exponent form (e.g., 2² -> 2**2)m
    text = re.sub(
        r"([0-9A-Za-z\)\]\}])([⁰¹²³⁴⁵⁶⁷⁸⁹⁺⁻]+)",
        lambda m: convert_superscripts(m.group(2), base=m.group(1)),
        text,
    )
    text = convert_superscripts(text)
    
    # numbers/roots
    text = text.replace("\\%", "%").replace("$", "").replace("%", "")
    text = re.sub(
        r"\\sqrt\s*\{([^}]*)\}",
        lambda match: f"sqrt({match.group(1)})",
        text,
    )
    text = re.sub(
        r"\\sqrt\s+([^\\\s{}]+)",
        lambda match: f"sqrt({match.group(1)})",
        text,
    )

    # fractions
    text = re.sub(
        r"\\frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}",
        lambda match: f"({match.group(1)})/({match.group(2)})",
        text,
    )
    text = re.sub(
        r"\\frac\s+([^\s{}]+)\s+([^\s{}]+)",
        lambda match: f"({match.group(1)})/({match.group(2)})",
        text,
    )

    # exponent and mixed numbers
    text = text.replace("^", "**")
    text = re.sub(
        r"(?<=\d)\s+(\d+/\d+)",
        lambda match: "+" + match.group(1),
        text,
    )

    # 1,234 -> 1234
    text = re.sub(
        r"(?<=\d),(?=\d\d\d(\D|$))",
        "",
        text,
    )

    # 남은 중괄호를 지우고 소문자로 맞춰 돌려줍니다
    return text.replace("{", "").replace("}", "").strip().lower()

In [ ]:
# 꺼내기 → 정규화를 이어서 해 봅니다
print(normalize_text(extract_final_candidate(model_answer)))

In [ ]:
# \dfrac과 $ 기호가 섞인 답을 정규화해 봅니다
print(normalize_text(r"$\dfrac{14}{3.}$"))

In [ ]:
print(normalize_text(r"\text{\[\frac{14}{3}\]}"))

In [ ]:
print(normalize_text("4/3"))

&nbsp;
## 3.6 수학적으로 같은지 확인하기

- 이 절에서는 꺼낸 답(모델이 만든 답)이 데이터셋에 있는 정답(ground truth)과 같은지 확인하는 기본 기능을 만듭니다. 5단계입니다.
- 다음 절(6단계)에서는 이 과정을 조금 더 튼튼하게 다듬어서 답이 맞는지 채점합니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch03/CH03_F08_raschka.webp?1" width="500px">

In [ ]:
# SymPy: 수식을 기호로 다루는 파이썬 라이브러리입니다
from sympy.parsing import sympy_parser as spp
from sympy.core.sympify import SympifyError
from tokenize import TokenError
from sympy.polys.polyerrors import PolynomialError

# 문자열 수식을 SymPy 수식으로 바꿉니다 (실패하면 None)
def sympy_parser(expr):
    # To avoid crashing on long garbage responses 
    # that some badly trained models (chapter 6) may emit
    if expr is None or len(expr) > 2000:
        return None
    try:
        return spp.parse_expr(
            expr,
            transformations=(
                # Standard transformations like handling parentheses
                *spp.standard_transformations,

                # Allow omitted multiplication symbols (e.g., "2x" -> 2*x")
                spp.implicit_multiplication_application,
            ),

            # Evaluate during parsing so simple constants simplify (e.g., 2+3 -> 5)
            evaluate=True,
        )
    # 파싱 중 날 수 있는 오류를 모두 잡아 None을 돌려줍니다
    except (SympifyError, SyntaxError, TypeError, AttributeError,
            IndexError, TokenError, ValueError, PolynomialError):
        return None

- 오류 처리가 지나치게 많아 보일 수 있습니다. 하지만 모두 MATH-500의 500문항 전체로 모델을 평가하면서 제가 실제로 만난 오류들입니다. 모델이 늘 깔끔한 형식으로 답을 내놓지는 않거든요.

In [ ]:
print(sympy_parser(normalize_text(
    extract_final_candidate(model_answer)
)))

In [ ]:
# 분수 문자열도 SymPy 수식으로 바뀝니다
print(sympy_parser("28/6"))

In [ ]:
from sympy import simplify

# 정답(ground truth)과 모델 답이 수학적으로 같은지 확인합니다
def equality_check(expr_gtruth, expr_pred):
    # First, check if the two expressions are exactly the same string
    if expr_gtruth == expr_pred:
        return True

    # Parse both expressions into SymPy objects (returns None if parsing fails)
    gtruth, pred = sympy_parser(expr_gtruth), sympy_parser(expr_pred)

    # If both expressions were parsed successfully, try symbolic comparison
    if gtruth is not None and pred is not None:
        try:
            # If the difference is 0, they are equivalent
            return simplify(gtruth - pred) == 0
        except (SympifyError, TypeError):
            pass

    return False

In [ ]:
# 끝 마침표와 괄호만 다른 같은 값
print(equality_check(
    normalize_text("13/4."),
    normalize_text(r"(13)/(4)")
))

In [ ]:
# 소수와 분수 비교
print(equality_check(
    normalize_text("0.5"),
    normalize_text(r"(1)/(2)")
))

In [ ]:
# 값이 다른 경우
print(equality_check(
    normalize_text("14/3"),
    normalize_text("15/3")
))

In [ ]:
# 순서쌍(튜플) 모양의 답 비교
print(equality_check(
    normalize_text("(14/3, 2/3)"),
    normalize_text("(14/3, 4/6)")
))

&nbsp;
## 3.7 답 채점하기

In [ ]:
# "(a, b)" 같은 순서쌍·목록 답을 부분별로 나눕니다
def split_into_parts(text):
    result = [text]

    if text:
        # Check if text looks like a tuple or list, e.g. "(a, b)" or "[a, b]"
        if (
            len(text) >= 2
            and text[0] in "([" and text[-1] in ")]"
            and "," in text[1:-1]
        ):
            # Split on commas inside brackets and strip whitespace
            items = [p.strip() for p in text[1:-1].split(",")]
            if all(items):
                result = items
    else:
        # If text is empty, return an empty list
        result = []

    return result

In [ ]:
split_into_parts(normalize_text(r"(14/3, 2/3)"))

In [ ]:
# 꺼낸 답(pred)과 정답(gt)을 정규화하고, 나누고, 부분마다 비교해 채점합니다
def grade_answer(pred_text, gt_text):
    result = False  # Default outcome if checks fail

    # Only continue if both inputs are non-empty strings
    if pred_text is not None and gt_text is not None:
        gt_parts = split_into_parts(
            normalize_text(gt_text)
        )  # Break ground truth into comparable parts

        pred_parts = split_into_parts(
            normalize_text(pred_text)
        )  # Break prediction into comparable parts

        # Ensure both sides have same number of valid parts
        if (gt_parts and pred_parts
           and len(gt_parts) == len(pred_parts)):
            result = all(
                equality_check(gt, pred)
                for gt, pred in zip(gt_parts, pred_parts)
            )  # Check each part for mathematical equivalence

    return result  # True only if all checks passed

In [ ]:
# 같은 값, 다른 표기
grade_answer("14/3", r"\frac{14}{3}")

In [ ]:
grade_answer(r"(14/3, 2/3)", "(14/3, 4/6)")

In [ ]:
# Define test cases: (name, prediction, ground truth, expected result)
tests = [
        ("check_1", "3/4", r"\frac{3}{4}", True),
        ("check_2", "(3)/(4)", r"3/4", True),
        ("check_3", r"\frac{\sqrt{8}}{2}", "sqrt(2)", True),
        ("check_4", r"\( \frac{1}{2} + \frac{1}{6} \)", "2/3", True),
        ("check_5", "(1, 2)", r"(1,2)", True),
        ("check_6", "(2, 1)", "(1, 2)", False),
        ("check_7", "(1, 2, 3)", "(1, 2)", False),
        ("check_8", "0.5", "1/2", True),
        ("check_9", "0.3333333333", "1/3", False),
        ("check_10", "1,234/2", "617", True),
        ("check_11", r"\text{2/3}", "2/3", True),
        ("check_12", "50%", "1/2", False),
        ("check_13", r"2\cdot 3/4", "3/2", True),
        ("check_14", r"90^\circ", "90", True),
        ("check_15", r"\left(\frac{3}{4}\right)", "3/4", True),
        ("check_16", r"2²", "2**2", True),
    ]


# 테스트마다 grade_answer를 돌려 기대값과 비교하고 표로 출력합니다
def run_demos_table(tests):
    header = ("Test", "Expect", "Got", "Status")
    rows = []
    for name, pred, gtruth, expect in tests:
        got = grade_answer(pred, gtruth)  # Run equality check
        # 기대한 결과와 같으면 PASS
        status = "PASS" if got == expect else "FAIL"
        rows.append((name, str(expect), str(got), status))

    data = [header] + rows
    
    # Compute max width for each column to align table nicely
    col_widths = [
        max(len(row[i]) for row in data)
        for i in range(len(header))
    ]

    # Print table row by row
    for row in data:
        line = " | ".join(
            row[i].ljust(col_widths[i])
            for i in range(len(header))
        )
        print(line)

    # Print summary of passed tests
    passed = sum(r[3] == "PASS" for r in rows)
    print(f"\nPassed {passed}/{len(rows)}")

In [ ]:
# 16개 테스트를 한 번에 돌립니다
run_demos_table(tests)

&nbsp;
## 3.8 평가 데이터셋 불러오기

- 앞 절에서 기본 평가 흐름(pipeline)을 만들었습니다.
- 이 절에서는 데이터셋을 불러옵니다(7단계). 다음 절에서 이 데이터셋에 평가 흐름을 적용해 모델을 평가합니다(8단계).

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch03/CH03_F09_raschka.webp?1" width="500px">

- 이 데이터셋은 [HuggingFaceH4/MATH-500](https://huggingface.co/datasets/HuggingFaceH4/MATH-500) 저장소에서 아래 코드로 내려받아 준비했습니다. 이 코드에는 [`datasets`](https://huggingface.co/docs/datasets/en/index) 패키지가 필요합니다(직접 실행할 필요는 없습니다. 참고용으로만 넣었습니다).

```python
from datasets import load_dataset
import json

dset = load_dataset("HuggingFaceH4/MATH-500", split="test")

math_data = dset.to_list()
with open("math500_test.json", "w", encoding="utf-8") as f:
    json.dump(math_data, f, ensure_ascii=False, indent=2)
```

In [ ]:
import json
import requests

# MATH-500 테스트 데이터를 불러옵니다 (로컬 파일이 없으면 깃허브에서 내려받기)
def load_math500_test(local_path="math500_test.json", save_copy=True):
    local_path = Path(local_path)
    url = (
        "https://raw.githubusercontent.com/rasbt/reasoning-from-scratch/"
        "main/ch03/01_main-chapter-code/math500_test.json"
    )

    if local_path.exists():
        with local_path.open("r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        data = r.json()

        if save_copy:  # Saves a local copy
            with local_path.open("w", encoding="utf-8") as f:
                json.dump(data, f, indent=2)

    return data

# 데이터를 불러와 문항 수를 확인합니다
math_data = load_math500_test()
print("Number of entries:", len(math_data))

In [ ]:
from pprint import pprint
# 첫 번째 문항이 어떻게 생겼는지 봅니다
pprint(math_data[0])

&nbsp;
## 3.9 모델 평가하기

- 앞 절에서 데이터셋을 불러왔습니다. 이제 이 데이터셋에 평가 흐름을 적용해 모델을 평가할 수 있습니다(7단계).

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch03/CH03_F10_raschka.webp?1" width="500px">

In [ ]:
# 문제를 프롬프트 템플릿에 넣어, 답을 \boxed{} 안에 쓰라고 시킵니다
def render_prompt(prompt):
    template = (
        "You are a helpful math assistant.\n"
        "Answer the question and write the final result on a new line as:\n"
        "\\boxed{ANSWER}\n\n"
        f"Question:\n{prompt}\n\nAnswer:"
    )
    return template

In [ ]:
prompt = (  # Same prompt we used at the beginning of the chapter
    r"If $a+b=3$ and $ab=\tfrac{13}{6}$, "
    r"what is the value of $a^2+b^2$?"
)
# 템플릿을 씌운 프롬프트를 확인합니다
prompt_fmt = render_prompt(prompt)
print(prompt_fmt)

In [ ]:
# 템플릿을 씌운 프롬프트로 다시 답을 만들어 봅니다
generated_text = generate_text_stream_concat(
    model, tokenizer, prompt_fmt, device,
    max_new_tokens=2048,
    verbose=True
)

In [ ]:
# 다른 프롬프트 템플릿 예시입니다 (문자열로 묶여 있어 실행되지 않습니다)
# Below is an alternative prompt template
# which swaps "Question" with "Problem"

"""
def render_prompt(prompt):
    template = (
        "You are a helpful math assistant.\n"
        "Solve the problem and write the final result on a new line as:\n"
        "\\boxed{ANSWER}\n\n"
        f"Problem:\n{prompt}\n\nAnswer:"
    )
    return template
"""

# This can noticeably affect the MATH-500 results:
# Base model on mps: improves accuracy 20% -> 40%
# Reasoning model on mps: worsens accuracy 90% -> 60%

In [ ]:
# 템플릿 없이 문제만 넣는 방법입니다 (이것도 실행되지 않는 예시입니다)
# Alternatively, we may use no prompt template

"""
def render_prompt(prompt):
    return prompt
"""

# This can noticeably affect the MATH-500 results:
# Base model on mps: improves accuracy 20% -> 70%
# Reasoning model on mps: worsens accuracy 90% -> 50%

In [ ]:
# 쉬운 문제 하나로 1~4단계 평가 흐름을 한 번에 시험해 봅니다
def mini_eval_demo(model, tokenizer, device):
    ex = {  # Test example with "problem" and "answer" fields
        "problem": "Compute 1/2 + 1/6.",
        "answer": "2/3"
    }
    prompt = render_prompt(ex["problem"])     # 1. Apply prompt template
    gen_text = generate_text_stream_concat(   # 2. Generate response
        model, tokenizer, prompt, device,
        max_new_tokens=64,
    )
    pred_answer = extract_final_candidate(gen_text)  # 3. Extract and normalize answer
    is_correct = grade_answer(                       # 4. Grade answer
        pred_answer, ex["answer"]
    )
    print(f"Device: {device}")
    print(f"Prediction: {pred_answer}")
    print(f"Ground truth: {ex['answer']}")
    print(f"Correct: {is_correct}")

In [ ]:
# 미니 평가를 돌려 봅니다
mini_eval_demo(model, tokenizer, device)

In [ ]:
import time


# Helper function to calculate remaining time
# 진행 상황과 남은 예상 시간(ETA)을 글로 만들어 줍니다
def eta_progress_message(
    processed,
    total,
    start_time,
    show_eta=False,
    label="Progress",
):
    progress = f"{label}: {processed}/{total}"
    pad_width = len(f"{label}: {total}/{total} | ETA: 00h 00m 00s")
    if not show_eta or processed <= 0:
        return progress.ljust(pad_width)

    # 지금까지 걸린 시간
    elapsed = time.time() - start_time
    if elapsed <= 0:
        return progress.ljust(pad_width)

    remaining = max(total - processed, 0)

    if processed:
        avg_time = elapsed / processed
        eta_seconds = avg_time * remaining
    else:
        eta_seconds = 0

    eta_seconds = max(int(round(eta_seconds)), 0)
    # 초를 시·분·초로 나눕니다
    minutes, rem_seconds = divmod(eta_seconds, 60)
    hours, minutes = divmod(minutes, 60)
    if hours:
        eta = f"{hours}h {minutes:02d}m {rem_seconds:02d}s"
    elif minutes:
        eta = f"{minutes:02d}m {rem_seconds:02d}s"
    else:
        eta = f"{rem_seconds:02d}s"

    message = f"{progress} | ETA: {eta}"
    return message.ljust(pad_width)

# MATH-500 문항을 하나씩 풀게 하고 채점해 정확도(accuracy)를 계산합니다
def evaluate_math500_stream(
    model,
    tokenizer,
    device,
    math_data,
    out_path=None,
    max_new_tokens=512,
    verbose=False,
):

    if out_path is None:
        dev_name = str(device).replace(":", "-")  # Make filename compatible with Windows
        out_path = Path(f"math500-{dev_name}.jsonl")

    num_examples = len(math_data)
    num_correct = 0
    start_time = time.time()

    with open(out_path, "w", encoding="utf-8") as f:  # Save results for inspection
        for i, row in enumerate(math_data, start=1):
            prompt = render_prompt(row["problem"])    # 1. Apply prompt template
            gen_text = generate_text_stream_concat(   # 2. Generate response
                model, tokenizer, prompt, device,
                max_new_tokens=max_new_tokens,
                verbose=verbose,
            )

            extracted = extract_final_candidate(  # 3. Extract and normalize answer
                gen_text
            )
            is_correct = grade_answer(            # 4. Grade answer
                extracted, row["answer"]
            )
            # 맞힌 개수를 셉니다
            num_correct += int(is_correct)

            record = {  # Record to be saved for inspection
                "index": i,
                "problem": row["problem"],
                "gtruth_answer": row["answer"],
                "generated_text": gen_text,
                "extracted": extracted,
                "correct": bool(is_correct),
            }
            # 한 줄에 한 문항씩 JSONL 파일로 저장합니다
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

            progress_msg = eta_progress_message(
                processed=i,
                total=num_examples,
                start_time=start_time,
                show_eta=True,
                label="MATH-500",
            )
            print(progress_msg, end="\r", flush=True)
            if verbose:  # Print responses during the generation process
                print(
                    f"\n\n{'='*50}\n{progress_msg}\n"
                    f"{'='*50}\nExtracted: {extracted}\n"
                    f"Expected:  {row['answer']}\n"
                    f"Correct so far: {num_correct}\n{'-'*50}"
                )

    # Print summary information
    seconds_elapsed = time.time() - start_time
    # 정확도 = 맞힌 개수 / 전체 문항 수
    acc = num_correct / num_examples if num_examples else 0.0
    print(f"\nAccuracy: {acc*100:.1f}% ({num_correct}/{num_examples})")
    print(f"Total time: {seconds_elapsed/60:.1f} min")
    print(f"Logs written to: {out_path}")
    return num_correct, num_examples, acc

- 데모용으로 10문항만 평가합니다(실행 시간을 적당히 줄이려고요).

In [ ]:
print("Model:", WHICH_MODEL)
print("Device:", device)
num_correct, num_examples, acc = evaluate_math500_stream(
    model, tokenizer, device, 
    # 데모용으로 앞 10문항만 씁니다 (전체는 math_data=math_data)
    math_data=math_data[:10],
    max_new_tokens=2048,
    verbose=False
)

| 모드      | 장치   | 정확도   | MATH-500 문항 수 | 걸린 시간             |
|-----------|--------|----------|---------------|-----------------------|
| Base      | CPU    | 30%      | 10            | 0.7 min (Mac Mini M4) |
| Base      | MPS    | 20%      | 10            | 0.4 min (Mac Mini M4) |
| Base      | CUDA   | 30%      | 10            | 0.2 min (DGX Spark)   |
| Base      | XPU    | 30%      | 10            | 1.2 min (Intel)       |
| Reasoning | CPU    | 90%      | 10            | 9.5 min (Mac Mini M4) |
| Reasoning | MPS    | 80%      | 10            | 3.8 min (Mac Mini M4) |
| Reasoning | CUDA   | 90%      | 10            | 3.7 min (DGX Spark)   |
| Reasoning | XPU    | 70%      | 10            | 8.5 min (Intel)       |


| 모드      | 장치   | 정확도   | MATH-500 문항 수 | 걸린 시간              |
|-----------|--------|----------|------------------|------------------------|
| Base      | CUDA   | 15.6%    | 500              | 10.0 min (DGX Spark)   |
| Reasoning | CUDA   | 48.2%    | 500              | 182.2 min (DGX Spark)  |

- 이 값들은 PyTorch 2.8에서 얻었습니다. PyTorch 버전이 다르면 달라질 수 있습니다.

- 참고로 위 표는 여러 환경에서 얻은 정확도(accuracy) 값입니다.
- 여기서 "GPU"는 NVIDIA("cuda") GPU를 말합니다. MPS는 애플 실리콘 M4 칩입니다.
- 추론 모델은 답을 훨씬 길게 쓰기 때문에 훨씬 느립니다.
- Qwen3-Base는 사전학습된 기본 모델이라, Qwen3 쪽에서는 채팅 템플릿(chat template) 없이 쓰기를 권합니다. 그런데 `tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)`를 `tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path, apply_chat_template=True)`로 바꾸면 MATH-500 성적이 크게 오릅니다(80%). 다만 MATH-500 테스트 세트가 학습 데이터에 들어 있었는지는 분명하지 않습니다. LLM 시대에는 인터넷에 있는 데이터라면 무엇이든 학습 데이터에 들어갔다고 보는 게 맞습니다([여기](https://github.com/rasbt/LLMs-from-scratch/pull/828#issuecomment-3324829736) 토론도 참고하세요).
- 500문항 전체를 돌리려면 `evaluate_math500_stream` 함수 호출에서 `math_data=math_data[:10],`를 `math_data=math_data,`로 바꾸면 됩니다.
- 보너스 자료에는 처리량을 높이려고 평가를 배치(batched) 방식으로 돌리는 스크립트가 있습니다([../02_math500-verifier-scripts/README.md](../02_math500-verifier-scripts/README.md) 참고). H100에서 배치 크기 128로 돌리면 기본 모델은 3.3분, 추론 모델은 14.6분에 평가할 수 있습니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch03/CH03_F11_raschka.webp?1" width="500px">

- 편하게 쓰고 싶다면 [../02_math500-verifier-scripts/evaluate_math500.py](../02_math500-verifier-scripts/evaluate_math500.py) 스크립트를 쓰세요. MATH-500 평가 코드를 명령줄에서 독립 스크립트로 돌려 줍니다(자세한 사용법은 [../02_math500-verifier-scripts/README.md](../02_math500-verifier-scripts/README.md)를 보세요).
- [../02_math500-verifier-scripts/evaluate_math500_batched.py](../02_math500-verifier-scripts/evaluate_math500_batched.py) 스크립트는 이 장의 코드를 배치 방식으로 돌립니다.
  - 한 번의 순전파(forward pass)에서 여러 문항을 함께 처리해 평가를 빠르게 합니다. 대신 메모리(RAM)가 더 필요합니다.
  - 배치 크기 128로 돌리면, H100 GPU에서 500문항 전체를 평가할 때 기본 모델 실행 시간이 13.3분에서 3.3분으로 줄어듭니다.
  - 마찬가지로 추론 모델은 500문항 기준 185.4분에서 14.6분으로 줄어듭니다.
  - H100은 예시일 뿐입니다. 이 스크립트는 다른 GPU(또는 CPU)에서도 돌아갑니다.

&nbsp;
## 정리

- 이 절에는 코드가 없습니다.